# 4.2 — Cleaning the Taxi Data

**Chapter 4, section 4.2 "Cleaning and Preprocessing"** — all four repairs, in the order the
chapter applies them — and the starting point for **Exercises 2 and 3**.

**The question this notebook answers:** ingestion delivered records; it did not deliver correct
ones. Four defects have to be repaired before anything downstream can be trusted, and each has
a mechanical part that Spark makes easy and a judgment part that no API can make for us:

1. **Missing values** — recognizing absence, then choosing deletion or imputation.
2. **Type conversion** — what Spark 4's ANSI defaults do to a bad cast, and how `try_cast`
   buys tolerance *with* a paper trail.
3. **Duplicate records** — why `dropDuplicates` is not enough when the duplicates are
   corrections.
4. **Formatting** — one country, five spellings; one payment type, four.

**Data.** The course taxi file is already clean: two million records with **no nulls at all**
in its numeric columns, no duplicates, and only five payment codes. Cleaning a clean file
teaches nothing, so the first section builds a *raw feed* from it: 200,000 records into which
this notebook injects each defect at a known rate. Knowing the rate is the point — every repair
below can be checked against the number of defects that were put in.

Runs on a laptop in about two minutes.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.2")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)   # this notebook catches errors on purpose

print("Spark", spark.version, "| ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

Spark 4.2.0 | ANSI mode: true


## 0. A raw feed with known defects

The injection is deterministic: every record is assigned a bucket from its own field values
with `hash`, and the defects are attached to particular buckets. No random seed, no dependence
on partitioning — the same feed is produced on every machine and every run.

| Bucket | Injected defect | Rate |
|--------|-----------------|------|
| 0, 1 | `trip_distance` set to null | 2 % |
| 5 | `trip_time` set to null | 1 % |
| 7 | `tip_amount` set to the sentinel `-1` | 1 % |
| 11, 12, 13 | `fare_text` respelled: `N/A`, a comma decimal, a leading space | 3 % |
| 20 | the record retransmitted three times, with a corrected `total_amount` | 1 % |
| all | `payment_type` respelled four ways per code | 100 % |

Only two of the three `fare_text` respellings are actually unconvertible, so the count that
section 2 recovers is **two** percent, not three: a leading space is tolerated by the number
parser. That is the first of this notebook's small surprises.

In [2]:
taxi_schema = StructType([
    StructField("medallion",         StringType(),    True),
    StructField("hack_license",      StringType(),    True),
    StructField("pickup_datetime",   TimestampType(), True),
    StructField("dropoff_datetime",  TimestampType(), True),
    StructField("trip_time",         IntegerType(),   True),
    StructField("trip_distance",     DoubleType(),    True),
    StructField("pickup_longitude",  DoubleType(),    True),
    StructField("pickup_latitude",   DoubleType(),    True),
    StructField("dropoff_longitude", DoubleType(),    True),
    StructField("dropoff_latitude",  DoubleType(),    True),
    StructField("payment_type",      StringType(),    True),
    StructField("fare_amount",       DoubleType(),    True),
    StructField("surcharge",         DoubleType(),    True),
    StructField("mta_tax",           DoubleType(),    True),
    StructField("tip_amount",        DoubleType(),    True),
    StructField("tolls_amount",      DoubleType(),    True),
    StructField("total_amount",      DoubleType(),    True),
])

N = 200_000
clean = (spark.read.schema(taxi_schema).option("header", "false")
         .csv(f"{DATA}/taxi-data-sorted-small.csv.bz2")
         .limit(N)                          # the file is time-sorted: this is the first ~2 days
         .drop("pickup_longitude", "pickup_latitude",
               "dropoff_longitude", "dropoff_latitude"))

bucket = F.pmod(F.hash("medallion", "pickup_datetime", "fare_amount"), F.lit(100))

spellings = {"CRD": ["CRD", "  CRD", "Credit ", "CREDIT"],
             "CSH": ["CSH", "CSH.", "  cash ", "CASH"]}
respelled = F.col("payment_type")
for code, variants in spellings.items():
    choice = F.element_at(F.array(*[F.lit(v) for v in variants]),
                          (F.pmod(F.col("bucket"), F.lit(4)) + 1).cast("int"))
    respelled = F.when(F.col("payment_type") == code, choice).otherwise(respelled)

feed = (clean
        .withColumn("bucket", bucket)
        .withColumn("ingested_at", F.col("pickup_datetime") + F.expr("INTERVAL 1 HOUR"))
        # 1. missing values
        .withColumn("trip_distance",
                    F.when(F.col("bucket") < 2, None).otherwise(F.col("trip_distance")))
        .withColumn("trip_time",
                    F.when(F.col("bucket") == 5, None).otherwise(F.col("trip_time")))
        # 2. a numeric sentinel that participates in arithmetic
        .withColumn("tip_amount",
                    F.when(F.col("bucket") == 7, F.lit(-1.0)).otherwise(F.col("tip_amount")))
        # 3. the money column as it arrives from the source: text
        .withColumn("fare_text",
                    F.when(F.col("bucket") == 11, F.lit("N/A"))
                     .when(F.col("bucket") == 12,
                           F.regexp_replace(F.col("fare_amount").cast("string"), r"\.", ","))
                     .when(F.col("bucket") == 13,
                           F.concat(F.lit(" "), F.col("fare_amount").cast("string")))
                     .otherwise(F.col("fare_amount").cast("string")))
        # 4. inconsistent spelling
        .withColumn("payment_type", respelled)
        .drop("fare_amount"))

# 5. retransmissions: bucket 20 arrives three times, later copies amending the total.
corrections = [feed.where(F.col("bucket") == 20)
               .withColumn("ingested_at", F.col("ingested_at") + F.expr(f"INTERVAL {k} HOUR"))
               .withColumn("total_amount", F.col("total_amount") + F.lit(k))
               for k in (1, 2)]
raw = feed.unionByName(corrections[0]).unionByName(corrections[1])

RAW = os.path.join(SCRATCH, "ch04-raw-feed")
raw.write.mode("overwrite").parquet(RAW)
raw = spark.read.parquet(RAW).cache()

n_raw = raw.count()
n_dup_extra = n_raw - N
print(f"{N:,} source records -> {n_raw:,} records in the feed "
      f"({n_dup_extra:,} of them retransmissions)")
raw.select("pickup_datetime", "trip_time", "trip_distance", "payment_type",
           "fare_text", "tip_amount", "total_amount").show(5, truncate=False)

200,000 source records -> 203,996 records in the feed (3,996 of them retransmissions)
+-------------------+---------+-------------+------------+---------+----------+------------+
|pickup_datetime    |trip_time|trip_distance|payment_type|fare_text|tip_amount|total_amount|
+-------------------+---------+-------------+------------+---------+----------+------------+
|2013-01-01 00:00:00|120      |0.44         |CASH        |3.5      |0.0       |4.5         |
|2013-01-01 00:02:00|0        |0.0          |CSH         |27.0     |0.0       |27.5        |
|2013-01-01 00:01:00|120      |0.71         |  cash      |4.0      |0.0       |5.0         |
|2013-01-01 00:01:00|120      |0.48         |CASH        |4.0      |0.0       |5.0         |
|2013-01-01 00:01:00|120      |0.61         |Credit      |4.0      |0.0       |5.0         |
+-------------------+---------+-------------+------------+---------+----------+------------+
only showing top 5 rows


## 1. Missing values

### Recognizing absence

The chapter's first point is that absence has several incompatible encodings and that, to
Spark, they are different things. A three-row table makes the distinction concrete before we
look for it in two hundred thousand.

In [3]:
encodings = spark.createDataFrame(
    [(1.0, "ok"), (None, ""), (float("nan"), None)],
    StructType([StructField("v", DoubleType()), StructField("s", StringType())]))

encodings.select(
    "v", "s",
    F.col("v").isNull().alias("v_isNull"),
    F.isnan("v").alias("v_isnan"),
    F.col("s").isNull().alias("s_isNull"),
    (F.col("s") == "").alias("s_is_empty")).show()

print("dropna() keeps", encodings.dropna().count(), "of", encodings.count(),
      "rows -- it treats NaN as missing too, though isNull does not.")

+----+----+--------+-------+--------+----------+
|   v|   s|v_isNull|v_isnan|s_isNull|s_is_empty|
+----+----+--------+-------+--------+----------+
| 1.0|  ok|   false|  false|   false|     false|
|NULL|    |    true|  false|   false|      true|
| NaN|NULL|   false|   true|    true|      NULL|
+----+----+--------+-------+--------+----------+

dropna() keeps 1 of 3 rows -- it treats NaN as missing too, though isNull does not.


In [4]:
# The sentinel is the dangerous one, because it is a valid number.
with_sentinel = raw.agg(F.avg("tip_amount").alias("mean_tip")).first()["mean_tip"]
as_null = (raw.withColumn("tip_amount",
                          F.when(F.col("tip_amount") == -1.0, None).otherwise(F.col("tip_amount")))
           .agg(F.avg("tip_amount").alias("mean_tip")).first()["mean_tip"])
print(f"mean tip with -1 treated as a number : ${with_sentinel:.4f}")
print(f"mean tip with -1 translated to null  : ${as_null:.4f}")
print(f"the sentinel moves the answer by      ${as_null - with_sentinel:.4f} on 1% of rows")

mean tip with -1 treated as a number : $1.0815
mean tip with -1 translated to null  : $1.1023
the sentinel moves the answer by      $0.0208 on 1% of rows


A one-percent sentinel moved the mean tip by about four cents. That is small enough to pass
review and large enough to be wrong, which is exactly why the chapter calls the in-band
sentinel the most dangerous encoding: it participates in arithmetic, and no null-check finds it.

### Deletion

`dropna` has three parameters and they answer different questions. The counts below are of the
same feed.

In [5]:
subset_cols = ["fare_text", "trip_distance"]
variants = [
    ("dropna()",                     raw.dropna()),
    ('dropna(how="all")',            raw.dropna(how="all")),
    ("dropna(thresh=12)",            raw.dropna(thresh=12)),
    (f"dropna(subset={subset_cols})", raw.dropna(subset=subset_cols)),
]
print(f"{'form':34s} {'rows kept':>10s} {'dropped':>8s}")
print(f"{'(the feed as it stands)':34s} {n_raw:>10,} {0:>8,}")
for label, d in variants:
    kept = d.count()
    print(f"{label:34s} {kept:>10,} {n_raw - kept:>8,}")

form                                rows kept  dropped
(the feed as it stands)               203,996        0
dropna()                              197,760    6,236


dropna(how="all")                     203,996        0
dropna(thresh=12)                     203,996        0
dropna(subset=['fare_text', 'trip_distance'])    199,901    4,095


`dropna()` and the `subset` form differ by about two thousand rows, and the difference is the
whole argument for the `subset` form: the rows it keeps have gaps only in columns this analysis
never reads. A blanket intolerance of nulls discards data for defects that do not matter.

Note also that **`how="all"` and `thresh=12` dropped nothing here**, because no record in this
feed is entirely or mostly empty. They are the right tools for a different defect, and reporting
zero is how you find that out — which is the rule the chapter states next: *a deletion step
should always be accompanied by a count of what it deleted.*

### Imputation, and the leakage that fit/transform prevents

In [6]:
# fillna selects columns by the type of its argument, which is easy to get wrong.
gaps = raw.select("payment_type", "trip_distance", "trip_time")
print("fillna('unknown') touches only string columns; fillna(0) only numeric ones:")
print("  nulls before        :",
      gaps.where(F.col("trip_distance").isNull()).count(), "in trip_distance")
print("  after fillna('none'):",
      gaps.fillna("none").where(F.col("trip_distance").isNull()).count(), "in trip_distance")
print("  after fillna(0)     :",
      gaps.fillna(0).where(F.col("trip_distance").isNull()).count(), "in trip_distance")

fillna('unknown') touches only string columns; fillna(0) only numeric ones:
  nulls before        : 4095 in trip_distance


  after fillna('none'): 4095 in trip_distance
  after fillna(0)     : 0 in trip_distance


In [7]:
from pyspark.ml.feature import Imputer

# Leakage, on five rows, where it can be seen. The training rows are 1, 2, 3, 100; the test
# rows are 5 and a gap. Fitting on the training set gives a median of 2; fitting on everything
# moves it, because the test row voted on the value the model will be trained with.
tiny_train = spark.createDataFrame([(1.0,), (2.0,), (3.0,), (100.0,), (None,)], "x double")
tiny_test  = spark.createDataFrame([(None,), (5.0,), (6.0,), (7.0,)], "x double")
fit_train = Imputer(inputCols=["x"], outputCols=["x_f"], strategy="median").fit(tiny_train)
fit_all   = Imputer(inputCols=["x"], outputCols=["x_f"], strategy="median") \
            .fit(tiny_train.unionByName(tiny_test))
print("median fitted on the training rows only:", fit_train.surrogateDF.first()["x"])
print("median fitted on training + test rows  :", fit_all.surrogateDF.first()["x"],
      " <- the test set has voted on it")

median fitted on the training rows only: 2.0
median fitted on training + test rows  : 5.0  <- the test set has voted on it


In [8]:
# Record what was imputed, before imputing it: one bit per row, and the analyst keeps the
# ability to ask later whether imputed rows behave differently.
flagged = (raw
           .withColumn("trip_distance_was_null", F.col("trip_distance").isNull())
           .withColumn("trip_time_was_null", F.col("trip_time").isNull()))

train, test = flagged.randomSplit([0.8, 0.2], seed=777)

imputer = Imputer(inputCols=["trip_distance", "trip_time"],
                  outputCols=["trip_distance_f", "trip_time_f"],
                  strategy="median")

model = imputer.fit(train)                 # statistics from the training set only
train_i = model.transform(train)
test_i  = model.transform(test)            # the same medians, applied to unseen data

leaky = imputer.fit(flagged)               # the mistake: fitted on everything

print("medians fitted on the training set only:", model.surrogateDF.first())
print("medians fitted on train + test         :", leaky.surrogateDF.first())
print()
print("rows imputed in the test set:",
      test_i.where("trip_distance_was_null OR trip_time_was_null").count())
test_i.where("trip_distance_was_null").select(
    "trip_distance", "trip_distance_f", "trip_distance_was_null").show(3)

medians fitted on the training set only: Row(trip_distance=2.06, trip_time=540.0)
medians fitted on train + test         : Row(trip_distance=2.06, trip_time=540.0)

rows imputed in the test set: 1228


+-------------+---------------+----------------------+
|trip_distance|trip_distance_f|trip_distance_was_null|
+-------------+---------------+----------------------+
|         NULL|           2.06|                  true|
|         NULL|           2.06|                  true|
|         NULL|           2.06|                  true|
+-------------+---------------+----------------------+
only showing top 3 rows


On the taxi feed the two medians happen to agree, because four thousand gaps spread evenly over
two hundred thousand rows do not move a median — and that is the ordinary case, which is what
makes the mistake so easy to keep. The five-row table above shows what is at stake when it does
move: the second fit *looked at the test set*, so every evaluation computed downstream would be
quoting a number that had already seen its own answer key. `Imputer` is an estimator: `fit`
learns the statistic, `transform` applies it, and the discipline is fit on training data,
transform on both.

Median rather than mean is also deliberate. Fare-like columns have long right tails, and a mean
would fill each gap with a value larger than most genuine observations.

## 2. Type conversion under Spark 4's ANSI defaults

`fare_text` is how the money column actually arrived: text. Three percent of it cannot be read
as a number. Under Spark 4 that is no longer a quiet matter.

In [9]:
# Every failure below is deliberate. A cast that fails on a 200,000-row column fails on an
# executor, which would otherwise flood this cell with a JVM stack trace, so quieten the log
# for the duration of the cell and restore it at the end.
spark.sparkContext.setLogLevel("FATAL")

def show_failure(label, fn):
    """Run something that is expected to raise, and print the error's own name for it."""
    try:
        fn()
        print(f"{label:44s} -> no error")
    except Exception as e:
        condition = getattr(e, "getCondition", lambda: "?")()
        print(f"{label:44s} -> {type(e).__name__}: {condition}")

show_failure("cast(fare_text AS double), 3% unconvertible",
             lambda: raw.select(F.col("fare_text").cast("double")).agg(F.sum("fare_text")).first())
show_failure("cast('abc' AS int)",     lambda: spark.sql("SELECT CAST('abc' AS INT)").first())
show_failure("cast(2147483648L AS int)", lambda: spark.sql("SELECT CAST(2147483648L AS INT)").first())
show_failure("2147483647 + 1",         lambda: spark.sql("SELECT 2147483647 + 1").first())
show_failure("1 / 0",                  lambda: spark.sql("SELECT 1/0").first())
show_failure("to_timestamp('nope')",
             lambda: spark.sql("SELECT to_timestamp('nope','yyyy-MM-dd HH:mm:ss')").first())

cast(fare_text AS double), 3% unconvertible  -> NumberFormatException: CAST_INVALID_INPUT
cast('abc' AS int)                           -> NumberFormatException: CAST_INVALID_INPUT
cast(2147483648L AS int)                     -> ArithmeticException: CAST_OVERFLOW
2147483647 + 1                               -> ArithmeticException: ARITHMETIC_OVERFLOW
1 / 0                                        -> ArithmeticException: DIVIDE_BY_ZERO
to_timestamp('nope')                         -> DateTimeException: CANNOT_PARSE_TIMESTAMP


In [10]:
# The error text is itself documentation: it names the try_ function that tolerates the failure.
try:
    raw.select(F.col("fare_text").cast("double")).agg(F.sum("fare_text")).first()
except Exception as e:
    message = " ".join(str(e).split())
    start = message.find("[CAST_INVALID_INPUT]")
    print(message[start:start + 320])
finally:
    spark.sparkContext.setLogLevel("ERROR")      # log level restored

[CAST_INVALID_INPUT] The value 'N/A' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018 == DataFrame == "cast" was called from line 3 in cell [11]


### Tolerance, accounted for

The `try_` family restores the permissive behaviour **expression by expression**, exactly where
bad data is expected, while everything else stays strict. The chapter insists the pattern be
imitated in full: `try_cast` alone reproduces Spark 3's silence, and it is the boolean column
and the count that make the difference.

In [11]:
tolerant = (raw
            .withColumn("fare_amount", F.col("fare_text").try_cast("double"))
            .withColumn("fare_was_bad",
                        F.col("fare_text").isNotNull() & F.col("fare_amount").isNull()))

audit = tolerant.select(
    F.count("*").alias("rows"),
    F.count(F.when(F.col("fare_was_bad"), 1)).alias("unconvertible"),
    F.round(F.avg(F.col("fare_was_bad").cast("int")) * 100, 2).alias("pct")).first()
print(f"{audit['unconvertible']:,} of {audit['rows']:,} fares could not be read "
      f"({audit['pct']}%), and the rows are still here to be inspected:")
tolerant.where("fare_was_bad").select("fare_text", "fare_amount", "total_amount").show(5)

4,028 of 203,996 fares could not be read (1.97%), and the rows are still here to be inspected:


+---------+-----------+------------+
|fare_text|fare_amount|total_amount|
+---------+-----------+------------+
|      N/A|       NULL|         8.3|
|      N/A|       NULL|         6.0|
|      4,0|       NULL|         5.0|
|      N/A|       NULL|        11.5|
|      3,5|       NULL|         4.5|
+---------+-----------+------------+
only showing top 5 rows


In [12]:
# Exercise 2(a): the four literals, under both regimes.
literals = spark.createDataFrame([("12.50",), ("N/A",), (" 9.99",), ("12,50",)], ["fare_text"])
print("try_cast (ANSI on, the default):")
literals.select("fare_text", F.col("fare_text").try_cast("double").alias("value")).show()

spark.conf.set("spark.sql.ansi.enabled", "false")
print("plain cast with ANSI disabled -- identical values, no error, no record of the failures:")
literals.select("fare_text", F.col("fare_text").cast("double").alias("value")).show()
spark.conf.set("spark.sql.ansi.enabled", "true")
print("ANSI mode restored:", spark.conf.get("spark.sql.ansi.enabled"))

try_cast (ANSI on, the default):
+---------+-----+
|fare_text|value|
+---------+-----+
|    12.50| 12.5|
|      N/A| NULL|
|     9.99| 9.99|
|    12,50| NULL|
+---------+-----+

plain cast with ANSI disabled -- identical values, no error, no record of the failures:


+---------+-----+
|fare_text|value|
+---------+-----+
|    12.50| 12.5|
|      N/A| NULL|
|     9.99| 9.99|
|    12,50| NULL|
+---------+-----+

ANSI mode restored: true


`" 9.99"` converts under both regimes — leading whitespace is tolerated by the parser — so only
two of the four literals are genuinely unconvertible. The comma decimal is the interesting one:
`"12,50"` is a perfectly good number in most of Europe and is not a number to Spark, which is a
reminder that a cast encodes a locale assumption nobody wrote down.

### The quiet failure: a cast that is entirely valid

In [13]:
# The course notebook's own conversion, which is legal under either regime.
spark.createDataFrame([(9.99,), (10.0,), (-9.99,), (29.999,)], ["fare"]) \
     .select("fare", F.col("fare").cast("int").alias("fare_category")).show()

bands = (tolerant.where("NOT fare_was_bad")
         .withColumn("fare_category", F.col("fare_amount").cast("int")))
print("the widest fares that share band 9, on the feed:")
(bands.where(F.col("fare_category") == 9)
      .agg(F.min("fare_amount").alias("cheapest_in_band_9"),
           F.max("fare_amount").alias("dearest_in_band_9"),
           F.count("*").alias("trips")).show())

+------+-------------+
|  fare|fare_category|
+------+-------------+
|  9.99|            9|
|  10.0|           10|
| -9.99|           -9|
|29.999|           29|
+------+-------------+

the widest fares that share band 9, on the feed:


+------------------+-----------------+-----+
|cheapest_in_band_9|dearest_in_band_9|trips|
+------------------+-----------------+-----+
|               9.0|              9.5|14190|
+------------------+-----------------+-----+



> **Two controls, not one.** ANSI mode governs conversions performed by *expressions in a
> query*. It does not govern what the reader does with a line it cannot parse; that was
> notebook [4.1](04.01%20Reader%20Policy%20and%20Corrupt%20Records.ipynb)'s `mode` option. The
> reader's mode decides which records exist; ANSI mode decides what their values may become.

## 3. Duplicate records

The feed contains retransmissions: one percent of records arrived three times, each copy with a
later `ingested_at` and an amended `total_amount`. This is the ordinary case, and it is the one
where `dropDuplicates` gives the right row count and the wrong rows.

In [14]:
keys = ["medallion", "pickup_datetime"]
print(f"records in the feed           : {n_raw:,}")
print(f"distinct() (all columns equal): {raw.distinct().count():,}"
      "   <- the copies differ, so this removes almost nothing")
print(f"dropDuplicates({keys}): {raw.dropDuplicates(keys).count():,}")
print(f"records injected as originals : {N:,}")

records in the feed           : 203,996


distinct() (all columns equal): 203,996   <- the copies differ, so this removes almost nothing


dropDuplicates(['medallion', 'pickup_datetime']): 200,000
records injected as originals : 200,000


In [15]:
# Which copy survives is arbitrary -- and "arbitrary" means it moves with the input order.
one_key = (raw.where(F.col("bucket") == 20)
           .select("medallion", "pickup_datetime", "ingested_at", "total_amount"))
sample_key = one_key.orderBy("ingested_at").first()
group = one_key.where((F.col("medallion") == sample_key["medallion"]) &
                      (F.col("pickup_datetime") == sample_key["pickup_datetime"]))
print("the three copies of one trip:")
group.orderBy("ingested_at").show(truncate=False)

for label, ordered in [("as stored",              group),
                       ("after orderBy(asc)",     group.orderBy("ingested_at")),
                       ("after orderBy(desc)",    group.orderBy(F.desc("ingested_at"))),
                       ("after repartition(3)",   group.repartition(3).orderBy(F.desc("total_amount")))]:
    survivor = ordered.dropDuplicates(keys).first()
    print(f"  dropDuplicates {label:22s} keeps total_amount = {survivor['total_amount']}")

the three copies of one trip:
+--------------------------------+-------------------+-------------------+------------+
|medallion                       |pickup_datetime    |ingested_at        |total_amount|
+--------------------------------+-------------------+-------------------+------------+
|0CA82E6914C3F20ECEF78298FA26F8C8|2013-01-01 00:01:00|2013-01-01 01:01:00|17.9        |
|0CA82E6914C3F20ECEF78298FA26F8C8|2013-01-01 00:01:00|2013-01-01 02:01:00|18.9        |
|0CA82E6914C3F20ECEF78298FA26F8C8|2013-01-01 00:01:00|2013-01-01 03:01:00|19.9        |
+--------------------------------+-------------------+-------------------+------------+

  dropDuplicates as stored              keeps total_amount = 17.9


  dropDuplicates after orderBy(asc)     keeps total_amount = 17.9


  dropDuplicates after orderBy(desc)    keeps total_amount = 19.9
  dropDuplicates after repartition(3)   keeps total_amount = 19.9


In [16]:
# Ranking inside a window states the intent exactly, and is reproducible.
w = Window.partitionBy(*keys).orderBy(F.col("ingested_at").desc())
deduplicated = (raw.withColumn("rn", F.row_number().over(w))
                   .filter("rn = 1")
                   .drop("rn"))
print(f"rows after window deduplication: {deduplicated.count():,} (originals: {N:,})")

latest = deduplicated.where((F.col("medallion") == sample_key["medallion"]) &
                            (F.col("pickup_datetime") == sample_key["pickup_datetime"])).first()
newest = group.orderBy(F.desc("ingested_at")).first()
assert latest["total_amount"] == newest["total_amount"], "the window kept the wrong copy"
print(f"the surviving copy is the latest correction: total_amount = {latest['total_amount']}"
      f" at {latest['ingested_at']}")

rows after window deduplication: 200,000 (originals: 200,000)
the surviving copy is the latest correction: total_amount = 19.9 at 2013-01-01 03:01:00


In [17]:
# Placement: deduplicate before the join, not after.
trips_small = raw.where(F.col("bucket") == 20)          # the retransmitted trips only
vehicles = trips_small.select("medallion").distinct().withColumn("make", F.lit("ford"))
vehicles_dirty = vehicles.unionByName(vehicles)         # every reference row twice

deduped_trips = (trips_small.withColumn("rn", F.row_number().over(w))
                 .filter("rn = 1").drop("rn"))

n_trips   = trips_small.count()
n_unique  = deduped_trips.count()
n_dirty   = trips_small.join(vehicles_dirty, "medallion", "left").count()
n_half    = deduped_trips.join(vehicles_dirty, "medallion", "left").count()
n_clean   = deduped_trips.join(vehicles_dirty.distinct(), "medallion", "left").count()
print(f"trips arriving (3 copies each)             : {n_trips:,}")
print(f"distinct trips                             : {n_unique:,}")
print()
print(f"both sides dirty  -> {n_dirty:,} rows   "
      f"(x{n_dirty / n_unique:.0f} the truth: 3 copies x 2 reference rows)")
print(f"trips cleaned only-> {n_half:,} rows   (x{n_half / n_unique:.0f})")
print(f"both sides cleaned-> {n_clean:,} rows   (x{n_clean / n_unique:.0f})")

trips arriving (3 copies each)             : 5,994
distinct trips                             : 1,998

both sides dirty  -> 11,988 rows   (x6 the truth: 3 copies x 2 reference rows)
trips cleaned only-> 3,996 rows   (x2)
both sides cleaned-> 1,998 rows   (x1)


A duplicated key does not repeat rows, it **multiplies** them: three copies on the left against
two on the right is six output rows where one belonged, which is exactly the factor above. Cleaning after the join means cleaning a
larger and stranger dataset than the one that was dirty, which is the chapter's placement rule
and the answer to Exercise 3(c).

Deduplication is a wide dependency — grouping equal keys together means a shuffle — so it is
also among the more expensive cleaning steps. Do it once, early, immediately after ingestion.

## 4. Formatting and standardization

The feed's `payment_type` holds two payment codes under eight spellings. Every `groupBy` over
that column reports eight groups where the truth has two.

In [18]:
raw.groupBy("payment_type").count().orderBy(F.desc("count")).show(truncate=False)

+------------+-----+
|payment_type|count|
+------------+-----+
|CSH         |31706|
|CSH.        |29466|
|CASH        |29340|
|  cash      |29332|
|CRD         |22065|
|Credit      |20901|
|  CRD       |20734|
|CREDIT      |20379|
|UNK         |73   |
+------------+-----+



In [19]:
# The mechanical layer: trim, then upper, then strip. The order is load-bearing.
chains = raw.select("payment_type").distinct().select(
    "payment_type",
    F.regexp_replace(F.upper(F.trim("payment_type")), r"[^A-Z]", "").alias("right_order"),
    F.trim(F.upper(F.regexp_replace("payment_type", r"[^A-Z]", ""))).alias("stripped_first"),
    F.regexp_replace(F.trim("payment_type"), r"[^A-Z]", "").alias("never_uppercased"))
chains.orderBy("payment_type").show(truncate=False)

+------------+-----------+--------------+----------------+
|payment_type|right_order|stripped_first|never_uppercased|
+------------+-----------+--------------+----------------+
|  CRD       |CRD        |CRD           |CRD             |
|  cash      |CASH       |              |                |
|CASH        |CASH       |CASH          |CASH            |
|CRD         |CRD        |CRD           |CRD             |
|CREDIT      |CREDIT     |CREDIT        |CREDIT          |
|CSH         |CSH        |CSH           |CSH             |
|CSH.        |CSH        |CSH           |CSH             |
|Credit      |CREDIT     |C             |C               |
|UNK         |UNK        |UNK           |UNK             |
+------------+-----------+--------------+----------------+



Both wrong orders destroy data rather than failing. Stripping before upper-casing deletes every
lowercase letter, so `"  cash "` becomes the empty string; never upper-casing does the same. The
correct chain is `trim`, then `upper`, then `regexp_replace`, and it collapses the nine
spellings in this feed to five.

In [20]:
# The semantic layer belongs in data, not code: a mapping table, applied as a left join.
mapping = spark.createDataFrame(
    [("CASH", "cash"), ("CSH", "cash"),
     ("CREDIT", "credit"), ("CRD", "credit")],
    ["raw", "canonical"])

mechanical = raw.withColumn(
    "payment_type", F.regexp_replace(F.upper(F.trim("payment_type")), r"[^A-Z]", ""))

standardized = (mechanical.join(mapping, mechanical.payment_type == mapping.raw, "left")
                .withColumn("payment_std", F.coalesce("canonical", "payment_type"))
                .drop("raw", "canonical"))

standardized.groupBy("payment_std").count().orderBy(F.desc("count")).show()
print("spellings the mapping table does not know, which survived the left join:")
(standardized.where(F.col("payment_std") == F.col("payment_type"))
 .groupBy("payment_std").count().show())

+-----------+------+
|payment_std| count|
+-----------+------+
|       cash|119844|
|     credit| 84079|
|        UNK|    73|
+-----------+------+

spellings the mapping table does not know, which survived the left join:


+-----------+-----+
|payment_std|count|
+-----------+-----+
|        UNK|   73|
+-----------+-----+



Two choices carry that listing. The join is `left`, so a spelling the table has never seen is
not discarded; and `coalesce` falls back to the mechanically cleaned value where no mapping
matched. `UNK`, `DIS` and `NOC` therefore come through **visibly**, where a profile can find
them, and each one found improves the table for every later run. (`coalesce` here is the column
function returning the first non-null of its arguments, not the `DataFrame.coalesce` that merges
partitions.)

## The cleaned feed, in the order the chapter applies the repairs

In [21]:
cleaned = (
    spark.read.parquet(RAW)
    # 1. sentinels become honest nulls, before anything averages them
    .withColumn("tip_amount",
                F.when(F.col("tip_amount") == -1.0, None).otherwise(F.col("tip_amount")))
    # 2. tolerated conversion, with the failure recorded
    .withColumn("fare_amount", F.col("fare_text").try_cast("double"))
    .withColumn("fare_was_bad",
                F.col("fare_text").isNotNull() & F.col("fare_amount").isNull())
    # 3. one row per trip, the latest correction winning
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
    # 4. one spelling per payment type
    .withColumn("payment_type",
                F.regexp_replace(F.upper(F.trim("payment_type")), r"[^A-Z]", ""))
).cache()

report = cleaned.select(
    F.count("*").alias("rows"),
    F.count(F.when(F.col("fare_was_bad"), 1)).alias("unconvertible_fares"),
    F.count(F.when(F.col("trip_distance").isNull(), 1)).alias("null_distance"),
    F.count(F.when(F.col("tip_amount").isNull(), 1)).alias("null_tip_was_sentinel"),
    F.count_distinct("payment_type").alias("payment_spellings")).first()
print(f"rows                       {report['rows']:,}   (source records: {N:,})")
print(f"unconvertible fares        {report['unconvertible_fares']:,}")
print(f"null trip_distance         {report['null_distance']:,}")
print(f"sentinels turned to null   {report['null_tip_was_sentinel']:,}")
print(f"distinct payment spellings {report['payment_spellings']}")

rows                       200,000   (source records: 200,000)
unconvertible fares        4,028
null trip_distance         4,095
sentinels turned to null   2,018
distinct payment spellings 5


Every number in that report was put into the feed on purpose, at a rate stated in section 0,
and every one came back out. That correspondence is the only evidence that a cleaning pipeline
does what its author believes.

## Conclusion

* **Absence has several encodings and Spark treats them as different things.** The in-band
  sentinel is the dangerous one: it is a valid number, no null-check finds it, and it moved the
  mean tip in this feed on one percent of rows.
* **Deletion is a decision about columns, not about nulls.** `subset` encodes the actual
  requirement; and every deletion step is reported, because dropping three rows is hygiene and
  dropping ten thousand is a finding.
* **Imputation statistics are fitted on training data only.** The alternative is a quiet
  leakage that inflates every later evaluation.
* **ANSI mode turns invalid casts into errors, and that is the improvement.** The correct
  response to expected bad data is `try_cast` *plus* the boolean column *plus* the count, not
  `spark.sql.ansi.enabled=false`.
* **A cast can also be entirely valid and still wrong**: `cast("int")` truncates, so a \$9.99
  fare is category 9. ANSI mode has nothing to say about that.
* **`dropDuplicates` chooses an arbitrary survivor**, and arbitrary means the choice moved three
  times in this notebook merely by reordering the input. A window ordering makes it deliberate.
* **Deduplicate before the join.** A duplicated key multiplies rather than repeats.
* **Standardize in two layers**: a mechanical chain in the right order, then a mapping table
  joined `left` with a `coalesce` fallback, so unknown spellings stay visible.

Next: [4.3](04.03%20Dates%20Strings%20and%20Reduction.ipynb) transforms the cleaned records.